In [ ]:
import os

from toffy.panel_utils import load_panel
from toffy.fov_watcher import start_watcher
from toffy.watcher_callbacks import build_callbacks

import shutil
from toffy import rosetta
from toffy.panel_utils import load_panel
from toffy.image_stitching import get_max_img_size
from alpineer.io_utils import list_folders
import pandas as pd
from operator import is_not
from functools import partial

from toffy import normalize
from toffy.panel_utils import load_panel
from alpineer.io_utils import list_folders

In [2]:
def find_channel_names(tmp_chan):
    core_compensation_channels=['Noodle_117','Noodle_120','Noodle_122','Noodle_118', 'Noodle_119','Noodle_121','Noodle_123','Noodle_124',
              'Xe_126','Xe_128','Xe_129','Xe_130','Xe_131','Xe_132','Xe_134','Ta metal','Gold']
    tmp_chan_mass=panel.loc[panel['Target']==tmp_chan, 'Mass'].item()

    #These lines are extracting the interferring target for known mass interferences. If there is no marker at the lower mass channel, then it is skipped.
    try:
        tmp_CN=panel.loc[panel['Mass']==(tmp_chan_mass-26), 'Target'].item()
    except:
        tmp_CN=None
    try:
        tmp_C2=panel.loc[panel['Mass']==(tmp_chan_mass-24), 'Target'].item()
    except:
        tmp_C2=None
    try:
        tmp_OH=panel.loc[panel['Mass']==(tmp_chan_mass-17), 'Target'].item()
    except:
        tmp_OH=None
    try:
        tmp_O=panel.loc[panel['Mass']==(tmp_chan_mass-16), 'Target'].item()
    except:
        tmp_O=None
    try:
        tmp_N=panel.loc[panel['Mass']==(tmp_chan_mass-14), 'Target'].item()
    except:
        tmp_N=None
    try:
        tmp_C=panel.loc[panel['Mass']==(tmp_chan_mass-12), 'Target'].item()
    except:
        tmp_C=None
    try:
        tmp_H=panel.loc[panel['Mass']==(tmp_chan_mass-1), 'Target'].item()
    except:
        tmp_H=None

    
    mass_interference_channel_names=[tmp_CN, tmp_C2, tmp_OH, tmp_O, tmp_N, tmp_C, tmp_H]
    mass_interference_channel_names=list(filter(partial(is_not, None), mass_interference_channel_names))
    compensation_channel_names=list(set(core_compensation_channels + mass_interference_channel_names))
    print(compensation_channel_names)
    return(compensation_channel_names)

In [30]:
def compensate_test_images(chans_to_correct):
    for tmp_chan_to_correct in chans_to_correct:
        print(tmp_chan_to_correct)
        #This checks if compensation channels have been manually set, or if the notebook needs to extract them
        compensation_channel_names=set_compensation_channel_names
        if compensation_channel_names is None:
            compensation_channel_names=find_channel_names(tmp_chan_to_correct)
        else:
            compensation_channel_names=set_compensation_channel_names
    
        #This will begin looping through the compensation channels for each specific marker and exporting those stitched images so that you can look at one channel at a time
        for tmp_chan_to_subtract in compensation_channel_names:
            
            # set multipliers
            multipliers =[0.5,1,5]
            # pick an informative name
            folder_name = tmp_chan_to_correct
        
            rosetta_mat_path = os.path.join(rosetta_testing_dir, cohort_name, matrix_file_name)
        
            # create sub-folder to hold images and files from this set of multipliers
            folder_path = os.path.join(rosetta_testing_dir, cohort_name, folder_name)
            if os.path.exists(folder_path):
            #    raise ValueError('This folder {} already exists, please' 
            #                     'pick a new name for each set of parameters'.format(folder_name))
                print(tmp_chan_to_correct+'-x-'+tmp_chan_to_subtract)
            else:
                os.makedirs(folder_path)
        
            # compensate the example fov images 
            rosetta.generate_rosetta_test_imgs(rosetta_mat_path, img_out_dir, multipliers, folder_path, 
                                               panel, tmp_chan_to_subtract, output_channel_names=[tmp_chan_to_correct])
        
            # stitch images together to enable easy visualization of outputs
            stitched_dir = os.path.join(folder_path, 'stitched_images')
            os.makedirs(stitched_dir, exist_ok=True)
        
            rosetta_dirs=[img_out_dir]
            for mult in multipliers:
                rosetta_dirs.append(os.path.join(folder_path, f'compensated_data_{mult}'))
        
            img_size = get_max_img_size(img_out_dir)
            rosetta.create_tiled_comparison(input_dir_list=rosetta_dirs, output_dir=stitched_dir, max_img_size=img_size, 
                                            channels=[tmp_chan_to_correct], img_sub_folder='rescaled')
        
            # add the source channel as first row to make evaluation easier
            #output_dir = os.path.join(rosetta_testing_dir, cohort_name, folder_name + '/' + folder_name + '-stitched_with_' + current_channel_name)
            output_dir = os.path.join(rosetta_testing_dir, cohort_name, folder_name + '/compensated-and-stitched' )
            os.makedirs(output_dir, exist_ok=True)
            rosetta.add_source_channel_to_tiled_image(raw_img_dir=img_out_dir, tiled_img_dir=stitched_dir,
                                                      output_dir=output_dir, source_channel=tmp_chan_to_subtract, max_img_size=img_size)
        
            # remove the intermediate compensated_data_{mult} and stitched_image dirs
            rosetta.clean_rosetta_test_dir(folder_path)

In [ ]:
#Set Directory
# the path to your home directory 

toffy_directory = 'C:\\Users\\esthe\\Documents\\toffy'

#include extension in panel_file_name
panel_file_name='Panel170_DK_Panel_Full_2025-06-25T15_34_57.183Z-toffy.csv'
json_run_metrics_folder_name='2025-04_DKMetab'

#path to your run folders
    #if they are all in one folder you can use this:
#run_names=os.listdir(os.path.join(home_directory, '/data/PROJ60-PAN71'))
    #if you want to list them manually use this:



In [ ]:
panel_path = os.path.join(toffy_directory, 'panel_files/',panel_file_name)
panel = load_panel(panel_path)
panel_meta = pd.read_csv(os.path.join(toffy_directory,"panel_files","Toffy_panel_meta.csv"))

extracted_imgs_dir = os.path.join(toffy_directory, 'Extracted_Images/')
run_names=list_folders(extracted_imgs_dir )                   
                     

rosetta_image_dir = os.path.join(toffy_directory, 'Rosetta_Compensated_Images')
rosetta_base_dir = os.path.join(toffy_directory, 'Rosetta_processing')
rosetta_matrix_dir = os.path.join(rosetta_base_dir, 'rosetta_matrices')
rosetta_testing_dir = os.path.join(rosetta_base_dir, 'rosetta_testing')

tuning_curve_file = 'avg_norm_func_2600.json'
autogain=False
bin_base_dir = os.path.join(toffy_directory,'run_metrics',json_run_metrics_folder_name + '_json')
normalized_base_dir = os.path.join(toffy_directory, 'Normalized_Images')

processed_base_dir = normalized_base_dir
cohort_image_dir = os.path.join(toffy_directory,'Cohorts')

print('Shared Paths')
print(extracted_imgs_dir)
print('Rosetta Paths')
print(rosetta_matrix_dir)
print(rosetta_image_dir)
print(rosetta_base_dir)
print(rosetta_testing_dir)
print('Normalization Paths')
print(bin_base_dir)
print(normalized_base_dir)

Panel170_DK_Panel_Full_2025-06-25T15_34_57.183Z-toffy.csv has the correct toffy format. Loading in panel data.
Shared Paths
C:\Users\esthe\Documents\toffy\Extracted_Images/
Rosetta Paths
C:\Users\esthe\Documents\toffy\Rosetta_processing\rosetta_matrices
C:\Users\esthe\Documents\toffy\Rosetta_Compensated_Images
C:\Users\esthe\Documents\toffy\Rosetta_processing
C:\Users\esthe\Documents\toffy\Rosetta_processing\rosetta_testing
Normalization Paths
C:\Users\esthe\Documents\toffy\run_metrics\2025-04_DKMetab_json
C:\Users\esthe\Documents\toffy\Normalized_Images


## 4a_compensate_image_data


In [10]:
#Name for your compensation folders
cohort_name='20260814'
#include extension in panel_file_name
matrix_file_name='offset_compensated_5.csv'

default_matrix_path = os.path.join(rosetta_base_dir, 'rosetta_matrices', matrix_file_name)
rosetta_mat_path = os.path.join(rosetta_testing_dir, cohort_name, matrix_file_name)

In [7]:
# copy random fovs from each run
rosetta.copy_image_files(cohort_name, run_names, rosetta_testing_dir, extracted_imgs_dir, fovs_per_run=1)

# copy rosetta matrix
shutil.copyfile(default_matrix_path, 
                os.path.join(rosetta_testing_dir, cohort_name, matrix_file_name))

# rescale images to allow direct comparison with rosetta
img_out_dir = os.path.join(rosetta_testing_dir, cohort_name, 'extracted_images')
rosetta.rescale_raw_imgs(img_out_dir)

Run just these lines of code to assign the correct img_out_dir path after uploading a new matrix file


In [15]:
shutil.copyfile(default_matrix_path, 
                os.path.join(rosetta_testing_dir, cohort_name, matrix_file_name))

# rescale images to allow direct comparison with rosetta
img_out_dir = os.path.join(rosetta_testing_dir, cohort_name, 'extracted_images')

rosetta.rescale_raw_imgs(img_out_dir)

Run one round of compensation on your base matrix to review all channels and see which ones you need to focus on:

In [ ]:
chans_to_correct =list(panel_meta[panel_meta.Type !="Background"]['Target'])
# pick the channel that you will be optimizing the coefficient for
current_channel_name = 'Noodle_117'

# set multipliers
multipliers = [1]

# pick an informative name
folder_name = 'Noodle_117'

rosetta_mat_path = os.path.join(rosetta_testing_dir, cohort_name, matrix_file_name)

# create sub-folder to hold images and files from this set of multipliers
folder_path = os.path.join(rosetta_testing_dir, cohort_name, folder_name)
if os.path.exists(folder_path):
    raise ValueError('This folder {} already exists, please ' 
                     'pick a new name for each set of parameters'.format(folder_name))
else:
    os.makedirs(folder_path)

# compensate the example fov images
rosetta.generate_rosetta_test_imgs(rosetta_mat_path, img_out_dir, multipliers, folder_path, 
                                   panel, current_channel_name, output_channel_names=None)

# stitch images together to enable easy visualization of outputs
stitched_dir = os.path.join(folder_path, 'stitched_images')
os.makedirs(stitched_dir)

rosetta_dirs=[img_out_dir]
for mult in multipliers:
    rosetta_dirs.append(os.path.join(folder_path, f'compensated_data_{mult}'))

img_size = get_max_img_size(img_out_dir)
scale = 0.5
rosetta.create_tiled_comparison(input_dir_list=rosetta_dirs, output_dir=stitched_dir, max_img_size=img_size, 
                                channels=None, img_size_scale=scale)

# add the source channel as first row to make evaluation easier
output_dir = os.path.join(rosetta_testing_dir, cohort_name, folder_name + '-stitched_with_' + current_channel_name)
os.makedirs(output_dir)
rosetta.add_source_channel_to_tiled_image(raw_img_dir=img_out_dir, tiled_img_dir=stitched_dir,
                                          output_dir=output_dir, source_channel=current_channel_name,
                                          max_img_size=img_size, img_size_scale=scale, img_sub_folder="rescaled")

# remove the intermediate compensated_data_{mult} and stitched_image dirs
rosetta.clean_rosetta_test_dir(folder_path)

Processing images with multiplier 1
Processing 2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT_fov-39-scan-1
Processing 2025-04-22-PROJ105-PAN170-SLD273_Set3-1_DKMetab-Bivona-WT_fov-19-scan-1
Processing 2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT_fov-17-scan-1


c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\toffy\rosetta.py:591: RuntimeWarning: divide by zero encountered in scalar divide
  perc_ratio = perc_source / perc_tile


Identify Markers/Channels that need to be cleaned up

In [29]:
## If you want to correct all channels unhash this line:

#tmp_chan_to_correct=list(panel_meta[panel_meta.Type !="Background"]['Target'])
#tmp_chan_to_correct = ['HK2','IDO1','LaminB1','CD66b','CD27','CD10','ATPIF1','dsDNA_Histone H3','TROP2','TTF1']
## If you want to manually choose which channel(s) to focus on:
#Even if you only want to focus on one channel, please keep it in list format: ['channel']
tmp_chan_to_correct=['HLA Class 1, ABC']

print(tmp_chan_to_correct)

['HLA Class 1, ABC']


#### Choose one option from below to optimize compensation coefficient for:


In [31]:
##This assigns compensation_channel_names as None so later on, we can extract the appropriate ones.
set_compensation_channel_names=None 

#or set manually 
#set_compensation_channel_names= ['dsDNA_Histone H3']

In [32]:
compensate_test_images(tmp_chan_to_correct)
# This will take a while, but will output folders for each marker that you are fine-tuning.
# Each folder will have tiffs for each of the channels you're correcting out

HLA Class 1, ABC
['CD45', 'Noodle_124', 'Noodle_121', 'Xe_128', 'FASN', 'DC-LAMP_LAMP3', 'CD27', 'Noodle_117', 'Xe_129', 'Ta metal', 'CD39_Biotin', 'Noodle_118', 'Xe_130', 'Noodle_120', 'Gold', 'Xe_132', 'CD3e', 'Xe_131', 'Noodle_119', 'CD14', 'Xe_134', 'Xe_126', 'Noodle_123', 'Noodle_122']
Processing images with multiplier 0.5
Processing 2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT_fov-13-scan-1
Processing 2025-04-21-PROJ105-PAN170-SLD272_Set2-1_DKMetab-Bivona-WT_fov-8-scan-1
Processing 2025-04-22-PROJ105-PAN170-SLD273_Set3-1_DKMetab-Bivona-WT_fov-17-scan-1
Processing 2025-04-22-PROJ105-PAN170-SLD273_Set3-1_DKMetab-Bivona-WT_fov-20-scan-1
Processing 2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT_fov-20-scan-1
Processing 2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT_fov-22-scan-1
Processing images with multiplier 1
Processing 2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT_fov-13-scan-1
Processing 2025-04-21-PROJ105-PAN170-SLD272_Set2-1_DKM

#Final compensation

In [33]:
# provide the matrix file name
final_matrix_name = 'offset_compensated_5.csv'
# this folder will hold the post-rosetta images

final_rosetta_path = os.path.join(rosetta_matrix_dir, final_matrix_name)

# if you would like to process all of the run folders in the image dir instead of just the runs tested, you can use the below line
runs = os.listdir(extracted_imgs_dir)

print(final_rosetta_path)
print(runs)


C:\Users\esthe\Documents\toffy\Rosetta_processing\rosetta_matrices\offset_compensated_5.csv
['2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT', '2025-04-17-PROJ105-PAN170-SLD271_Set1-2_DKMetab-Bivona-WT', '2025-04-17-PROJ105-PAN170-SLD271_Set1-3_DKMetab-Bivona-WT', '2025-04-21-PROJ105-PAN170-SLD272_Set2-1_DKMetab-Bivona-WT', '2025-04-22-PROJ105-PAN170-SLD272_Set2-2_DKMetab-Bivona-WT', '2025-04-22-PROJ105-PAN170-SLD273_Set3-1_DKMetab-Bivona-WT', '2025-04-22-PROJ105-PAN170-SLD273_Set3-2_DKMetab-Bivona-WT', '2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT']


In [34]:
#target_mass = list(panel_meta[panel_meta.Type !="Background"]['Mass'])
for run in runs:
    print("processing run {}".format(run))
    if not os.path.exists(os.path.join(rosetta_image_dir, run)):
        os.makedirs(os.path.join(rosetta_image_dir, run))
    rosetta.compensate_image_data(raw_data_dir=os.path.join(extracted_imgs_dir, run), 
                                  comp_data_dir=os.path.join(rosetta_image_dir, run), 
                                  comp_mat_path=final_rosetta_path,  panel_info=panel,  batch_size=1)

processing run 2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT
Processing fov-10-scan-1
Processing fov-11-scan-1
Processing fov-12-scan-1
Processing fov-13-scan-1
Processing fov-14-scan-1
Processing fov-15-scan-1
Processing fov-17-scan-1
Processing fov-18-scan-1
Processing fov-19-scan-1
Processing fov-2-scan-1
Processing fov-20-scan-1
Processing fov-21-scan-1
Processing fov-22-scan-1
Processing fov-23-scan-1
Processing fov-24-scan-1
Processing fov-25-scan-1
Processing fov-26-scan-1
Processing fov-27-scan-1
Processing fov-28-scan-1
Processing fov-3-scan-1
Processing fov-30-scan-1
Processing fov-31-scan-1
Processing fov-32-scan-1
Processing fov-33-scan-1
Processing fov-34-scan-1
Processing fov-35-scan-1
Processing fov-36-scan-1
Processing fov-37-scan-1
Processing fov-38-scan-1
Processing fov-39-scan-1
Processing fov-4-scan-1
Processing fov-40-scan-1
Processing fov-41-scan-1
Processing fov-42-scan-1
Processing fov-43-scan-1
Processing fov-44-scan-1
Processing fov-5-scan-1
Proces

#Normalized images

In [35]:

for run_name in runs:
    mph_run_dir = os.path.join(toffy_directory,'run_metrics',json_run_metrics_folder_name + '_run-metrics', run_name, 'fov_data')

    # check the run for changes in detector voltage
    normalize.check_detector_voltage(os.path.join(bin_base_dir, run_name))

    # specify sub-folder for rosetta images
    img_sub_folder = 'rescaled'

    # create directory to hold normalized images
    normalized_run_dir = os.path.join(normalized_base_dir, run_name)
    if not os.path.exists(normalized_run_dir):
        os.makedirs(normalized_run_dir)

    # create directory to hold associated processing files
    if not os.path.exists(mph_run_dir):
        os.makedirs(mph_run_dir)
        
    # plot the voltage change across FOVs if autogain set
    # otherwise, verify the voltage across all FOVs is constant
    if autogain:
        normalize.plot_detector_voltage(
            run_folder=os.path.join(bin_base_dir, run_name),
            mph_run_dir=os.path.dirname(mph_run_dir)
        )
    else:
        normalize.check_detector_voltage(os.path.join(bin_base_dir, run_name))

    # get all FOVs
    fovs = list_folders(os.path.join(rosetta_image_dir, run_name), substrs='fov-')

    # loop over each FOV
    for fov in fovs:
        # generate mph values
        mph_file_path = os.path.join(mph_run_dir, fov + '_pulse_heights.csv')
        if not os.path.exists(mph_file_path):
            normalize.write_mph_per_mass(base_dir=os.path.join(bin_base_dir, run_name), output_dir=mph_run_dir, 
                                         fov=fov, 
                                         masses= panel['Mass'].values,
                                         start_offset=0.3, stop_offset=0)

    normalize.normalize_image_data(img_dir=os.path.join(rosetta_image_dir, run_name), norm_dir=normalized_run_dir, pulse_height_dir=mph_run_dir,
                               panel_info= panel,
                               img_sub_folder=img_sub_folder,
                               norm_func_path=os.path.join(toffy_directory, 'tuning_curves', tuning_curve_file),
                               autogain=autogain)

c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\toffy\normalize.py:242: UserWarning: Removing previously generated combined pulse_heights file in C:\Users\esthe\Documents\toffy\run_metrics\2025-04_DKMetab_run-metrics\2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT\fov_data
  warnings.warn(
c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\toffy\normalize.py:242: UserWarning: Removing previously generated combined pulse_heights file in C:\Users\esthe\Documents\toffy\run_metrics\2025-04_DKMetab_run-metrics\2025-04-17-PROJ105-PAN170-SLD271_Set1-2_DKMetab-Bivona-WT\fov_data
  warnings.warn(
c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\toffy\normalize.py:242: UserWarning: Removing previously generated combined pulse_heights file in C:\Users\esthe\Documents\toffy\run_metrics\2025-04_DKMetab_run-metrics\2025-04-17-PROJ105-PAN170-SLD271_Set1-3_DKMetab-Bivona-WT\fov_data
  warnings.warn(
c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\toffy

## 5_Reorganization Notebook


In [ ]:
import os

from toffy import reorg
from alpineer.io_utils import list_folders

In [ ]:
cohort_name = '2026Aug_renormalized'
#Check the run_names loaded at the top of the script are correct
run_names

['2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT',
 '2025-04-17-PROJ105-PAN170-SLD271_Set1-2_DKMetab-Bivona-WT',
 '2025-04-17-PROJ105-PAN170-SLD271_Set1-3_DKMetab-Bivona-WT',
 '2025-04-21-PROJ105-PAN170-SLD272_Set2-1_DKMetab-Bivona-WT',
 '2025-04-22-PROJ105-PAN170-SLD272_Set2-2_DKMetab-Bivona-WT',
 '2025-04-22-PROJ105-PAN170-SLD273_Set3-1_DKMetab-Bivona-WT',
 '2025-04-22-PROJ105-PAN170-SLD273_Set3-2_DKMetab-Bivona-WT',
 '2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT']

In [ ]:
cohort_path = os.path.join(cohort_image_dir, cohort_name)
if not os.path.exists(cohort_path):
    os.makedirs(cohort_path)

In [40]:
# rename FOVs in each of the runs in run_names
reorg.rename_fovs_in_cohort(run_names=run_names, processed_base_dir=processed_base_dir, 
                            cohort_path=cohort_path, bin_base_dir=bin_base_dir)

Renaming FOVs in 2025-04-16-PROJ105-PAN170-SLD271_Set1-1_DKMetab-Bivona-WT


c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\alpineer\misc_utils.py:136: UserWarning: Not all values given in list fovs in run file were found in list existing fov folders.
 Displaying 10 of 14 invalid value(s) for list fovs in run file
1            fov-1-scan-1
2            fov-1-scan-2
3            fov-1-scan-3
4            fov-1-scan-4
5            fov-1-scan-5
6            fov-16-scan-1
7            fov-16-scan-2
8            fov-16-scan-3
9            fov-29-scan-1
10           fov-29-scan-2

  warnings.warn(err_str)


Renaming FOVs in 2025-04-17-PROJ105-PAN170-SLD271_Set1-2_DKMetab-Bivona-WT


c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\alpineer\misc_utils.py:136: UserWarning: Not all values given in list fovs in run file were found in list existing fov folders.
 Displaying 10 of 25 invalid value(s) for list fovs in run file
1            fov-1-scan-1
2            fov-1-scan-2
3            fov-1-scan-3
4            fov-1-scan-4
5            fov-1-scan-5
6            fov-6-scan-1
7            fov-7-scan-1
8            fov-8-scan-1
9            fov-9-scan-1
10           fov-10-scan-1

  warnings.warn(err_str)


Renaming FOVs in 2025-04-17-PROJ105-PAN170-SLD271_Set1-3_DKMetab-Bivona-WT


...

c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\alpineer\misc_utils.py:136: UserWarning: Not all values given in list fovs in run file were found in list existing fov folders.
 Displaying 10 of 11 invalid value(s) for list fovs in run file
1            fov-1-scan-1
2            fov-1-scan-2
3            fov-1-scan-3
4            fov-1-scan-4
5            fov-1-scan-5
6            fov-16-scan-1
7            fov-16-scan-2
8            fov-16-scan-3
9            fov-25-scan-1
10           fov-25-scan-2

  warnings.warn(err_str)


Renaming FOVs in 2025-04-22-PROJ105-PAN170-SLD273_Set3-2_DKMetab-Bivona-WT


c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\alpineer\misc_utils.py:136: UserWarning: Not all values given in list fovs in run file were found in list existing fov folders.
 Displaying 6 of 6 invalid value(s) for list fovs in run file
1            fov-13-scan-1
2            fov-13-scan-2
3            fov-13-scan-3
4            fov-27-scan-1
5            fov-27-scan-2
6            fov-27-scan-3

  warnings.warn(err_str)


Renaming FOVs in 2025-04-23-PROJ105-PAN170-SLD274_Set4-1_DKMetab-Bivona-WT


c:\Users\esthe\miniconda3\envs\toffy_env\Lib\site-packages\alpineer\misc_utils.py:136: UserWarning: Not all values given in list fovs in run file were found in list existing fov folders.
 Displaying 10 of 11 invalid value(s) for list fovs in run file
1            fov-1-scan-1
2            fov-1-scan-2
3            fov-1-scan-3
4            fov-1-scan-4
5            fov-1-scan-5
6            fov-16-scan-1
7            fov-16-scan-2
8            fov-16-scan-3
9            fov-31-scan-1
10           fov-31-scan-2

  warnings.warn(err_str)


## 3. Creating a Single Cohort (Optional)
Once all of the FOVs within each folder have been renamed and all of the partial runs have been combined together, you can now get rid of the run structure and create a single cohort directory of FOVS. The function below will combine all of the FOVs within each of your distinct runs into a single directory with the run name prepended. 

**For example, if you have a structure like this:**

*  20220101_run_1
    *  tonsil_1
    *  tonsil_2
*  20220102_run_2
    *  lymph_1
    *  spleen_2

**It will get merged into something that looks like this:**
* image_data
    *  20220101_run_1_tonsil_1
    *  20220101_run_1_tonsil_2
    *  20220102_run_2_lymph_1
    *  20220102_run_2_spleen_2

**This is not required; if you plan on processing each run separately, such as for tiled images, you can skip this step. However, if you will be doing all of your analysis at the individual FOV level, this will simplify the downstream steps.**

In [41]:
reorg.combine_runs(cohort_dir=cohort_path)

## QC Longitudinal Control Metrics
In order to make use of the QC Longitudinal Control Metrics, you need to have run `5_rename_and_reorganize.ipynb` beforehand. There is no set naming convention here for each FOV.

To make use of the LC Metrics, your cohort should consist of one control sample across several runs

```sh
my_control_sample_runs/
├── MY_CONTROL_SAMPLE_RUN1/
├── MY_CONTROL_SAMPLE_RUN2/
├── ...
├── MY_CONTROL_SAMPLE_RUN5/
├── MY_CONTROL_SAMPLE_RUN6/
└── MY_CONTROL_SAMPLE_RUN7/
```

Longitudinal Control Metrics can be computed for control sample FOVs across different cohorts. For a given control sample, we will be able to analyze it's run-to-run variance. 

In [43]:
import os
from pathlib import Path
from toffy import qc_comp, qc_metrics_plots

In [44]:
new_cohort_path = Path(cohort_path) / "image_data"

In [ ]:
#Remove run-names in the folder

for folder in new_cohort_path.iterdir():
    if not folder.is_dir():
        continue

    new_name = folder.name
    for s in runs:
        new_name = new_name.replace(s, "")
        

    # Skip if nothing changed
    if new_name == folder.name:
        continue

    folder.rename(folder.with_name(new_name))
    print(f"{folder.name} -> {new_name}")

In [49]:
#remove underbar
for folder in new_cohort_path.iterdir():
    if folder.is_dir() and folder.name.startswith("_"):
        new_name = folder.name[1:]  # Remove the first character
        new_name = new_name.replace("_duplicate1","")
        folder.rename(folder.with_name(new_name))
        print(f"{folder.name} -> {new_name}")

In [51]:
qc_metrics_path = Path(cohort_path) / "qc_metrics"
qc_metrics = ["Non-zero mean intensity"]

#Compute QC for each patient ID ; Run TH016 and TH169 separately
patient_IDs = ['AZ003','AZ025','AZ036','AZ043','AZ052','TH101','TH102','TH107','Tonsil']
channel_exclude = list(panel_meta[panel_meta.Type =="Background"]['Target'])
channel_include = list(panel_meta[panel_meta.Type !="Background"]['Target'])



In [52]:
#patient_IDs = ['TH107','Tonsil']
for patient_ID in patient_IDs:
    qc_metrics_dir = Path(qc_metrics_path) / f"qc_metrics_{patient_ID}"
    qc_metrics_dir.mkdir(parents=True, exist_ok=True)

    fovs = [
        p.name
        for p in new_cohort_path.iterdir()
        if p.is_dir() and patient_ID in p.name
    ]

    qc_control = qc_comp.QCControlMetrics(
        qc_metrics=qc_metrics,
        cohort_path=str(new_cohort_path),
        metrics_dir=str(qc_metrics_dir),
    )

    qc_control.compute_control_qc_metrics(
        control_sample_name=patient_ID,
        fovs=fovs,
        channel_exclude = channel_exclude,
        channel_include=channel_include,
    )

    #generate and save heatmap
    qc_metrics_plots.longitudinal_control_heatmap(
        qc_control=qc_control, control_sample_name=patient_ID, save_figure=True, dpi=300
    )

Computing QC Longitudinal Control metrics - AZ003:   0%|          | 0/27 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - AZ025:   0%|          | 0/14 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - AZ036:   0%|          | 0/19 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - AZ043:   0%|          | 0/21 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - AZ052:   0%|          | 0/22 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - TH101:   0%|          | 0/27 [00:00<?, ?FOVs/s]

Computing QC Longitudinal Control metrics - TH102:   0%|          | 0/21 [00:00<?, ?FOVs/s]

ValueError: Index contains duplicate entries, cannot reshape

In [53]:

qc_metrics_dir = Path(qc_metrics_path) / "TH016_169"
qc_metrics_dir.mkdir(parents=True, exist_ok=True)

fovs = ['TH169-E6-B1_FOV01','TH016-E3-B1_FOV01','TH016-E3-B1_FOV02']

qc_control = qc_comp.QCControlMetrics(
        qc_metrics=qc_metrics,
        cohort_path=str(new_cohort_path),
        metrics_dir=str(qc_metrics_dir),
    )

qc_control.compute_control_qc_metrics(
        control_sample_name= "TH016_169",
        fovs=fovs,
        channel_exclude = channel_exclude,
        channel_include=channel_include,
    )

    #generate and save heatmap
qc_metrics_plots.longitudinal_control_heatmap(
        qc_control=qc_control, control_sample_name= "TH016_169", save_figure=True, dpi=300
    )

Computing QC Longitudinal Control metrics - TH016_169:   0%|          | 0/3 [00:00<?, ?FOVs/s]

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\esthe\\Documents\\toffy\\Cohorts\\2026Aug_renormalized\\image_data\\TH016-E3-B1_FOV01\\'